In [153]:
import pandas as pd
import numpy as np
import os
import sys
from datetime import timedelta
from utils import get_data_path
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import plotly.graph_objects as go
from xgboost import XGBRegressor

# Setup Paths
notebook_dir = os.path.dirname(os.getcwd())  
project_root = os.path.dirname(notebook_dir) 
sys.path.append(project_root)  

# Load Data
df = pd.read_csv(get_data_path('data/data_raw.csv'))
df['StartDateTime'] = pd.to_datetime(df['StartDateTime'], format='%d/%m/%Y %H:%M')
df.set_index('StartDateTime', inplace=True)
df = df.asfreq('30min')  # setting frequency 

# NaNs 
print("Number of NaNs in data:", df['ISEM DA Price'].isna().sum())

Number of NaNs in data: 0


### data prep and setting features

#### overview
Takes raw electricity price data and prepares it for forecasting by creating relevant features. Uses most recent 12 months of data with 30-minute intervals.

#### how the Code Works:
Data Loading: Loads CSV, converts timestamps to datetime index
Time Window: Filters to last 12 months using cutoff_date
Feature Creation: Creates multiple feature types for prediction

#### feature types:
Time Features: hour, day_of_week, month, is_weekend, quarter_of_day, Peak_Hour
External Features: DemandForecast-DAM, WindForecast-DAM, Fuel.Gas, Fuel.Carbon
Price Lag Features: Previous period, same hour yesterday, same hour last week
Calculated Features: Gas_to_Wind_Ratio (gas price divided by wind forecast)

#### feature engineering:
Points_per_day: Automatically determines if data is 30min (48 points) or hourly (24 points)
Peak_Hour: hours between 7:00-22:00 as peak demand periods
Gas_to_Wind_Ratio: relationship between gas prices and wind generation
Add features by extending the 'features' list
Modify time windows by adjusting cutoff_date
Adjust lag periods by changing shift values

In [154]:
# Load Data
df = pd.read_csv(get_data_path('data/data_raw.csv'))
df['StartDateTime'] = pd.to_datetime(df['StartDateTime'], format='%d/%m/%Y %H:%M')
df.set_index('StartDateTime', inplace=True)
df = df.asfreq('30min')  # Ensure frequency is set

cutoff_date = df.index[-1] - pd.DateOffset(months=12)

df_recent = df[df.index >= cutoff_date].copy()
df_recent = df_recent.sort_index()
time_delta = df_recent.index[1] - df_recent.index[0]
print(f"Time delta between points: {time_delta}")

external_features = [
    # Original features
    'DemandForecast-DAM', 'WindForecast-DAM', 'Fuel.Gas', 'Fuel.Carbon',
    
    # New features
    'GB DAM N2EX Price',        # Current GB Prices
    'NetDemandForecast-DAM',    # Net Demand (Demand minus Wind)
    'DAM Unavailability'        # Outages
]

# Add GB forecast as a separate feature - it's already for the next day
df_recent['GB_Tomorrow_Price_FC'] = df_recent['GB DA N2EX Price FC']

# Basic time features
df_recent['hour'] = df_recent.index.hour
df_recent['day_of_week'] = df_recent.index.dayofweek
df_recent['month'] = df_recent.index.month
df_recent['is_weekend'] = df_recent.index.dayofweek >= 5
df_recent['quarter_of_day'] = df_recent.index.hour // 6

# Original lag features
df_recent['Price_Lag1'] = df_recent['ISEM DA Price'].shift(1)  # Previous period
df_recent['Price_Lag_SameHourYesterday'] = df_recent['ISEM DA Price'].shift(points_per_day)  # same hour yesterday
df_recent['Price_Lag_SameHourLastWeek'] = df_recent['ISEM DA Price'].shift(points_per_day * 7)  # same hour last week

# Keep the original calculated features
df_recent['Gas_to_Wind_Ratio'] = df_recent['Fuel.Gas'] / df_recent['WindForecast-DAM'].clip(lower=0.1)
df_recent['Peak_Hour'] = ((df_recent.index.hour >= 7) & (df_recent.index.hour <= 22)).astype(int)

# Final features list
features = external_features + [
    'hour', 'day_of_week', 'month', 'is_weekend', 'quarter_of_day', 'Peak_Hour',
    'Price_Lag1', 'Price_Lag_SameHourYesterday', 'Price_Lag_SameHourLastWeek',
    'Gas_to_Wind_Ratio', 'GB_Tomorrow_Price_FC'
]

def smape(actual, predicted):
    return 100 * np.mean(2 * np.abs(predicted - actual) / (np.abs(actual) + np.abs(predicted) + 1e-10))




Time delta between points: 0 days 00:30:00


#### train/test splot, training + preds

trains XGBoost regressor on 80% of recent data (12 months), evals on 20% test set. 
uses hyperparameters suggested by AI and trains the model using XGBoost and uses squared error as the objective over 200 trees

after that is some outputs for important features, and the plot

In [155]:
# Train/test split
train_size = int(len(df_recent) * 0.8)
train = df_recent.iloc[:train_size]
test = df_recent.iloc[train_size:]
print(f"Training data: {len(train)} points ({train.index.min()} to {train.index.max()})")
print(f"Testing data: {len(test)} points ({test.index.min()} to {test.index.max()})")

# Prepare data matrices
X_train = train[features]
y_train = train['ISEM DA Price']
X_test = test[features]
y_test = test['ISEM DA Price']

# Define and train model
xgb_model = XGBRegressor(
    objective='reg:squarederror',
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=2,
    colsample_bytree=0.8,
    subsample=0.8,
    random_state=42
)

print("Training XGBoost model...")
xgb_model.fit(X_train, y_train)

# Generate predictions
train_preds = xgb_model.predict(X_train)
test_preds = xgb_model.predict(X_test)

# Calculate metrics for test data (out-of-sample)
rmse = np.sqrt(mean_squared_error(y_test, test_preds))
mae = mean_absolute_error(y_test, test_preds)
smape_val = smape(y_test, test_preds)
r2 = r2_score(y_test, test_preds)

print("\nXGBoost Model Performance (Test Set):")
print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")
print(f"sMAPE: {smape_val:.2f}%")
print(f"R² Score: {r2:.4f}")

# Calculate metrics for negative prices specifically
neg_price_idx = y_test < 0
if sum(neg_price_idx) > 0:
    neg_price_rmse = np.sqrt(mean_squared_error(y_test[neg_price_idx], test_preds[neg_price_idx]))
    print(f"RMSE on negative prices: {neg_price_rmse:.2f}")

# Feature importance analysis
importance = pd.DataFrame({
    'Feature': features,
    'Importance': xgb_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("\nTop 5 most important features:")
print(importance.head(5))


Training data: 14055 points (2024-01-20 23:30:00 to 2024-11-08 18:30:00)
Testing data: 3514 points (2024-11-08 19:00:00 to 2025-01-20 23:30:00)
Training XGBoost model...

XGBoost Model Performance (Test Set):
RMSE: 20.88
MAE: 10.91
sMAPE: 10.75%
R² Score: 0.9004
RMSE on negative prices: 3.44

Top 5 most important features:
                  Feature  Importance
13             Price_Lag1    0.553605
12              Peak_Hour    0.108900
5   NetDemandForecast-DAM    0.077755
4       GB DAM N2EX Price    0.060357
16      Gas_to_Wind_Ratio    0.035656


In [156]:
train_preds_series = pd.Series(train_preds, index=train.index)
test_preds_series = pd.Series(test_preds, index=test.index)

# Create plot
fig = go.Figure()

# Training data
fig.add_trace(go.Scatter(
    x=train.index,
    y=train['ISEM DA Price'],
    name='Training Data',
    line=dict(color='blue')
))

# Test data
fig.add_trace(go.Scatter(
    x=test.index,
    y=test['ISEM DA Price'],
    name=' Test Values',
    line=dict(color='green')
))

# Out-of-sample forecast
fig.add_trace(go.Scatter(
    x=test.index,
    y=test_preds_series,
    name='XGBoost Forecast',
    line=dict(color='red', dash='dash')
))

# Add vertical line for train/test split
split_date = test.index[0]
fig.add_vline(x=split_date, line_width=2, line_dash="dash", line_color="black")

# Layout
fig.update_layout(
    title="XGBoost Electricity Price Forecast",
    xaxis_title="Date",
    yaxis_title="Price (£/MWh)",
    hovermode="x unified",
    legend=dict(orientation="h", y=1.1)
)

# Range selector - FIX: Change "week" to "day" with count=7
fig.update_xaxes(
    rangeslider_visible=True,
    rangeselector=dict(
        buttons=list([
            dict(count=1, label="1m", step="month", stepmode="backward"),
            dict(count=3, label="3m", step="month", stepmode="backward"),
            dict(count=7, label="1w", step="day", stepmode="backward"),  # Changed from "week" to "day" with count=7
            dict(step="all")
        ])
    )
)

# Show plot
fig.show()

#### Forecast

Uses trained XGBoost model to forecast next trading day (48 points from 23:00-23:00) 
sets up forecast timeframe, abd created 3 data series (last week actuak price, model pred for last week + next day forecast)
and visulisases all of it

In [157]:
# set up forecast period
last_date = df_recent.index.max().date()
forecast_start = pd.Timestamp(last_date).replace(hour=23, minute=0)
forecast_end = forecast_start + timedelta(hours=24)
forecast_index = pd.date_range(start=forecast_start, end=forecast_end - timedelta(minutes=30), freq='30min')

forecast_df = pd.DataFrame(index=forecast_index)

# Basic time features
forecast_df['hour'] = forecast_df.index.hour
forecast_df['day_of_week'] = forecast_df.index.dayofweek
forecast_df['month'] = forecast_df.index.month
forecast_df['is_weekend'] = forecast_df.index.dayofweek >= 5
forecast_df['quarter_of_day'] = forecast_df.index.hour // 6
forecast_df['Peak_Hour'] = ((forecast_df.index.hour >= 7) & (forecast_df.index.hour <= 22)).astype(int)

# Set external features to their last known values
for feature in external_features:
    forecast_df[feature] = df_recent[feature].iloc[-1]

forecast_df['GB_Tomorrow_Price_FC'] = df_recent['GB DA N2EX Price FC'].iloc[-1]
forecast_df['Price_Lag1'] = df_recent['ISEM DA Price'].iloc[-1]
forecast_df['Price_Lag_SameHourYesterday'] = df_recent['ISEM DA Price'].iloc[-48:].values
forecast_df['Price_Lag_SameHourLastWeek'] = df_recent['ISEM DA Price'].iloc[-336:-288].values

forecast_df['Gas_to_Wind_Ratio'] = forecast_df['Fuel.Gas'] / forecast_df['WindForecast-DAM'].clip(lower=0.1)

# generate forecast
forecast_prices = xgb_model.predict(forecast_df[features])

# get last week of actual prices and their predictions
week_start = forecast_start - timedelta(days=7)
actual_week = df_recent.loc[week_start:forecast_start, 'ISEM DA Price']

# get model predictions for the same period
historical_forecast = xgb_model.predict(df_recent.loc[week_start:forecast_start, features])
historical_forecast_series = pd.Series(historical_forecast, index=actual_week.index)

# make plot
fig3 = go.Figure()

#  last week of actual prices
fig3.add_trace(go.Scatter(
    x=actual_week.index,
    y=actual_week,
    name='Actual Prices (Last Week)',
    line=dict(color='green')
))

#   model predictions for last week
fig3.add_trace(go.Scatter(
    x=actual_week.index,
    y=historical_forecast_series,
    name='Model Predictions (Last Week)',
    line=dict(color='red', dash='dash')
))

#  next day forecast
fig3.add_trace(go.Scatter(
    x=forecast_index,
    y=forecast_prices,
    name='Next Day Forecast',
    line=dict(color='blue')
))

fig3.add_vline(x=forecast_start, line_width=2, line_dash="dash", line_color="black")
fig3.update_layout(
    title="Last Week Actuals + Predictions + Next Day Forecast",
    xaxis_title="Date",
    yaxis_title="Price (£/MWh)",
    hovermode="x unified",
    legend=dict(orientation="h", y=1.1)
)

fig3.show()

# save forecast
forecast_output = pd.DataFrame({
    'ISEM DA Forecast': forecast_prices
}, index=forecast_index)

# Save to CSV
os.makedirs('data/xgboost', exist_ok=True)  # Create directory if it doesn't exist
forecast_output.to_csv('data/xgboost/xgboost_forecast.csv')

print("\nForecast saved to 'data/xgboost/xgboost_forecast.csv'")
print("\nForecast Summary:")
print(f"Average forecast: £{forecast_prices.mean():.2f}/MWh")
print(f"Min forecast: £{forecast_prices.min():.2f}/MWh at {forecast_index[forecast_prices.argmin()]}")
print(f"Max forecast: £{forecast_prices.max():.2f}/MWh at {forecast_index[forecast_prices.argmax()]}")



Forecast saved to 'data/xgboost/xgboost_forecast.csv'

Forecast Summary:
Average forecast: £132.38/MWh
Min forecast: £115.83/MWh at 2025-01-20 23:00:00
Max forecast: £148.96/MWh at 2025-01-21 08:00:00


In [158]:
print("\nNext 24-Hour XGBoost Forecast (Every 30 Minutes):")
for timestamp, price in zip(forecast_index, forecast_prices):
    print(f"{timestamp} : £{price:.2f}")



Next 24-Hour XGBoost Forecast (Every 30 Minutes):
2025-01-20 23:00:00 : £115.83
2025-01-20 23:30:00 : £115.83
2025-01-21 00:00:00 : £126.26
2025-01-21 00:30:00 : £126.26
2025-01-21 01:00:00 : £125.79
2025-01-21 01:30:00 : £125.79
2025-01-21 02:00:00 : £127.46
2025-01-21 02:30:00 : £127.46
2025-01-21 03:00:00 : £125.55
2025-01-21 03:30:00 : £125.55
2025-01-21 04:00:00 : £127.15
2025-01-21 04:30:00 : £127.15
2025-01-21 05:00:00 : £130.24
2025-01-21 05:30:00 : £130.24
2025-01-21 06:00:00 : £128.91
2025-01-21 06:30:00 : £128.91
2025-01-21 07:00:00 : £146.74
2025-01-21 07:30:00 : £146.74
2025-01-21 08:00:00 : £148.96
2025-01-21 08:30:00 : £148.96
2025-01-21 09:00:00 : £141.66
2025-01-21 09:30:00 : £141.66
2025-01-21 10:00:00 : £137.03
2025-01-21 10:30:00 : £137.03
2025-01-21 11:00:00 : £136.85
2025-01-21 11:30:00 : £136.85
2025-01-21 12:00:00 : £137.59
2025-01-21 12:30:00 : £137.59
2025-01-21 13:00:00 : £137.24
2025-01-21 13:30:00 : £137.24
2025-01-21 14:00:00 : £137.24
2025-01-21 14:30:00

In [159]:
# Add residuals to test set
residuals = abs(test['ISEM DA Price'] - test_preds_series)
test_with_residuals = test.copy()
test_with_residuals['Absolute Error'] = residuals
test_with_residuals['hour'] = test_with_residuals.index.hour

# Group by hour of the day
hourly_mae = test_with_residuals.groupby('hour')['Absolute Error'].mean()

# Plot
import plotly.graph_objects as go

fig = go.Figure(data=go.Bar(x=hourly_mae.index, y=hourly_mae.values))
fig.update_layout(
    title='Mean Absolute Error by Hour of the Day',
    xaxis_title='Hour (0 = Midnight)',
    yaxis_title='MAE (£)',
    xaxis=dict(tickmode='linear', dtick=1),
    bargap=0.1
)
fig.show()


In [160]:
# Extract hour from forecast index
forecast_df = pd.DataFrame({
    'Forecast Price': forecast_prices
}, index=forecast_index)
forecast_df['hour'] = forecast_df.index.hour

# Group by hour
avg_forecast_by_hour = forecast_df.groupby('hour')['Forecast Price'].mean()

# Plot
fig2 = go.Figure(data=go.Bar(x=avg_forecast_by_hour.index, y=avg_forecast_by_hour.values))
fig2.update_layout(
    title='Average Forecasted Price by Hour',
    xaxis_title='Hour of Day',
    yaxis_title='Average Forecasted Price (£)',
    xaxis=dict(tickmode='linear', dtick=1),
    bargap=0.1
)
fig2.show()


In [161]:
summary = pd.DataFrame({
    'Hour': hourly_mae.index,
    'MAE (£)': hourly_mae.values.round(2),
    'Avg Forecasted Price (£)': avg_forecast_by_hour.values.round(2)
})

print(summary.to_string(index=False))


 Hour  MAE (£)  Avg Forecasted Price (£)
    0     5.81                126.260002
    1     4.06                125.790001
    2     3.36                127.459999
    3     3.77                125.550003
    4     3.69                127.150002
    5     4.70                130.240005
    6     7.12                128.910004
    7    11.51                146.740005
    8    13.05                148.960007
    9    14.28                141.660004
   10    12.50                137.029999
   11    11.48                136.850006
   12    11.30                137.589996
   13    12.14                137.240005
   14    12.25                137.240005
   15    14.20                139.889999
   16    19.14                141.410004
   17    24.27                135.490005
   18    19.45                131.990005
   19    12.61                126.320000
   20    11.45                126.339996
   21    10.46                125.839996
   22     8.17                119.269997
   23    11.10  

In [162]:
# Add rolling standard deviation to capture local volatility (e.g. over 24 hours = 48 points)
df_recent['Rolling_Std_24h'] = df_recent['ISEM DA Price'].rolling(window=48).std()
# Drop NaNs (due to rolling window)
volatility_corr_df = df_recent.dropna(subset=['Rolling_Std_24h'])

# Correlation with external factors
external_vars = ['DemandForecast-DAM', 'WindForecast-DAM', 'Fuel.Gas', 'Fuel.Carbon']
correlations = volatility_corr_df[external_vars + ['Rolling_Std_24h']].corr()

# Show correlation of volatility with each factor
print("Correlation with Volatility (Rolling Std):")
print(correlations['Rolling_Std_24h'].sort_values(ascending=False))
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_recent.index,
    y=df_recent['Rolling_Std_24h'],
    name='Rolling Std (24h)',
    line=dict(color='orange')
))

fig.update_layout(
    title='Electricity Price Volatility (24h Rolling Standard Deviation)',
    xaxis_title='Date',
    yaxis_title='Volatility (£)',
    hovermode='x unified'
)

fig.show()


Correlation with Volatility (Rolling Std):
Rolling_Std_24h       1.000000
Fuel.Gas              0.426206
DemandForecast-DAM    0.280144
Fuel.Carbon           0.062115
WindForecast-DAM     -0.086802
Name: Rolling_Std_24h, dtype: float64


### 19th JANUARY 2025 FORECAST + COMPARISON TO ACTUAL

In [163]:
last_date = df_recent.index.max().date()
target_date = last_date - timedelta(days=2)
forecast_start = pd.Timestamp(target_date).replace(hour=23, minute=0)
forecast_end = forecast_start + timedelta(hours=24)
forecast_index = pd.date_range(start=forecast_start, end=forecast_end - timedelta(minutes=30), freq='30min')
forecast_df = pd.DataFrame(index=forecast_index)
jan_19_date = pd.Timestamp('2025-01-19').date()
jan_19_data = df[df.index.date == jan_19_date]
jan_18_date = pd.Timestamp('2025-01-18').date()
jan_18_data = df[df.index.date == jan_18_date]
jan_18_last_two_hours = jan_18_data[jan_18_data.index.hour >= 22]

forecast_df['hour'] = forecast_df.index.hour
forecast_df['day_of_week'] = forecast_df.index.dayofweek
forecast_df['month'] = forecast_df.index.month
forecast_df['is_weekend'] = forecast_df.index.dayofweek >= 5
forecast_df['quarter_of_day'] = forecast_df.index.hour // 6
forecast_df['Peak_Hour'] = ((forecast_df.index.hour >= 7) & (forecast_df.index.hour <= 22)).astype(int)
for feature in external_features:
    forecast_df[feature] = jan_19_data[feature].values if not jan_19_data.empty else df_recent[feature].iloc[-1]

# using the features that are AVAILABLE TO POWERNI ON THE DATE OF 19TH JAN 
forecast_df['GB_Tomorrow_Price_FC'] = jan_19_data['GB DA N2EX Price FC'].values[0] if not jan_19_data.empty else df_recent['GB DA N2EX Price FC'].iloc[-1]
forecast_df['Price_Lag1'] = df_recent['ISEM DA Price'].iloc[-1]
forecast_df['Price_Lag_SameHourYesterday'] = df_recent['ISEM DA Price'].iloc[-48:].values
forecast_df['Price_Lag_SameHourLastWeek'] = df_recent['ISEM DA Price'].iloc[-336:-288].values
forecast_df['Gas_to_Wind_Ratio'] = forecast_df['Fuel.Gas'] / forecast_df['WindForecast-DAM'].clip(lower=0.1)
forecast_prices = xgb_model.predict(forecast_df[features])
week_start = forecast_start - timedelta(days=7)
actual_week = df_recent.loc[week_start:forecast_start, 'ISEM DA Price']
historical_forecast = xgb_model.predict(df_recent.loc[week_start:forecast_start, features])
historical_forecast_series = pd.Series(historical_forecast, index=actual_week.index)

fig3 = go.Figure()
fig3.add_trace(go.Scatter(x=actual_week.index, y=actual_week, name='Actual Prices (Last Week)', line=dict(color='green')))
fig3.add_trace(go.Scatter(x=actual_week.index, y=historical_forecast_series, name='Model Predictions (Last Week)', line=dict(color='red', dash='dash')))
fig3.add_trace(go.Scatter(x=forecast_index, y=forecast_prices, name='January 19th Forecast', line=dict(color='blue')))
fig3.add_vline(x=forecast_start, line_width=2, line_dash="dash", line_color="black")
fig3.update_layout(title="Last Week Actuals + Predictions + January 19th Forecast", xaxis_title="Date", yaxis_title="Price (£/MWh)", 
    hovermode="x unified", legend=dict(orientation="h", y=1.1))
fig3.show()

forecast_output = pd.DataFrame({'ISEM DA Forecast': forecast_prices}, index=forecast_index)
os.makedirs('data/xgboost', exist_ok=True)
forecast_output.to_csv('data/xgboost/xgboost_forecast_jan19.csv')
print(f"\nForecast saved to 'data/xgboost/xgboost_forecast_jan19.csv'")
print(f"\nJanuary 19th Forecast Summary:")
print(f"Average forecast: £{forecast_prices.mean():.2f}/MWh")
print(f"Min forecast: £{forecast_prices.min():.2f}/MWh at {forecast_index[forecast_prices.argmin()]}")
print(f"Max forecast: £{forecast_prices.max():.2f}/MWh at {forecast_index[forecast_prices.argmax()]}")

jan_19_actual = jan_19_data['ISEM DA Price']
common_forecast = forecast_prices[:min(len(forecast_prices), len(jan_19_actual))]
mae = mean_absolute_error(jan_19_actual[:len(common_forecast)], common_forecast)
print(f"\nModel accuracy - MAE: £{mae:.2f}/MWh")

jan_19_actual = pd.concat([jan_18_last_two_hours['ISEM DA Price'], jan_19_data['ISEM DA Price']])
jan_19_actual = jan_19_actual[jan_19_actual.index.isin(forecast_index)]
comparison_df = pd.DataFrame({'Actual': jan_19_actual, 'Forecast': forecast_prices, 'Error': jan_19_actual.values - forecast_prices}, 
    index=jan_19_actual.index)
comparison_df.to_csv('data/xgboost/jan19_comparison.csv')



Forecast saved to 'data/xgboost/xgboost_forecast_jan19.csv'

January 19th Forecast Summary:
Average forecast: £123.83/MWh
Min forecast: £96.25/MWh at 2025-01-19 01:00:00
Max forecast: £162.63/MWh at 2025-01-19 15:00:00

Model accuracy - MAE: £12.08/MWh
